# Data Exploration

In [1]:
RAW_PATH = "../../data/raw/Software.jsonl"
CHUNK_SIZE = 5_000
MAX_CHUNKS_FOR_SAMPLE = 3  

In [2]:
#imports
import json
import re
import random
from collections import Counter
from pathlib import Path
import pandas as pd

In [3]:
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", None)

In [4]:
path = Path(RAW_PATH)
assert path.exists(), f"File not found: {path.resolve()}"   # The .resolve() part converts the relative path into the full absolute path so the error message tells you exactly where Python looked.

## Step 1 — Row count + schema

In [5]:
total_rows = 0
all_columns_seen = set()
key_set_counter = Counter()
chunk_count = 0

In [6]:
reader = pd.read_json(path, lines=True, chunksize=CHUNK_SIZE)

In [7]:
def _is_present(v):
    # Check if the input value `v` is either a list or a dictionary.
    # If it is, we consider it "present" regardless of whether it's empty or not.
    if isinstance(v, (list, dict)):
        return True

    # For all other data types (like strings, numbers, etc.),
    # use Pandas' `notna` function to check if the value is not missing.
    # `pd.notna(v)` returns True if `v` is not NaN, None, or NaT.
    return pd.notna(v)


In [8]:
for chunk in reader:
    chunk_count += 1
    total_rows += len(chunk)
    all_columns_seen |= set(chunk.columns) # Union of sets to track all columns seen across chunks
    # |= union assignment operator is used to update the set of all columns seen with the columns from the current chunk.
    # Eg.: If `all_columns_seen` was {'A', 'B'} and the current chunk has columns {'B', 'C'}, after this operation, `all_columns_seen` will be {'A', 'B', 'C'}.

    # Iterate over each row in the DataFrame, converted to a dictionary.
    # `chunk.to_dict(orient="records")` produces a list of dicts,
    # where each dict represents one row: {column_name: value}.
    for record in chunk.to_dict(orient="records"):

        # Build a set of column names (keys) that have "present" values in this row.
        # `_is_present(v)` is used to decide if a value counts as present.
        # `frozenset` is used instead of a normal set because it is immutable
        # and can be used as a dictionary key later.
        present_keys = frozenset(
                k for k, v in record.items()
                if _is_present(v)
            )
        
        # Increment the counter for this particular set of present keys.
        # `key_set_counter` is a Counter, so this line tracks how many rows share the same pattern of present columns.
        key_set_counter[present_keys] += 1


print(f"Chunks read: {chunk_count}  (chunk size: {CHUNK_SIZE:,})")
print(f"Total rows: {total_rows:,}")
print(f"\nColumns seen across all chunks: {sorted(all_columns_seen)}")
print(f"\nDistinct key-sets found: {len(key_set_counter)}  (1 = fully consistent schema)")

Chunks read: 977  (chunk size: 5,000)
Total rows: 4,880,181

Columns seen across all chunks: ['asin', 'helpful_vote', 'images', 'parent_asin', 'rating', 'text', 'timestamp', 'title', 'user_id', 'verified_purchase']

Distinct key-sets found: 1  (1 = fully consistent schema)


## Column Check - Review/Metadata ?

In [9]:
review_like = {"text", "reviewtext", "rating", "overall", "user_id", "reviewerid"}
metadata_like = {"title", "brand", "price", "description", "rank", "main_cat"}

In [10]:
cols_lower = {c.lower() for c in all_columns_seen}

In [11]:
review_hits = cols_lower & review_like
meta_hits = cols_lower & metadata_like

In [12]:
if meta_hits and not review_hits:
    print("\n*** This looks like the METADATA file, not the reviews file. ***")

So far what we have done
- In the dataset there could be review related fields as well as mmetadata like fields.
- We have identified them and separate in "review_hits" and "meta_hits" sets.
- At last if "meta_hits" set has some fields and "review_hits" is empty that means given file is a metadata file and no review fields are present.

## If number of Distinct key-sets is greater than 1, Following code helps

In [13]:
print("Most common key-sets (top 5):")
for keys, count in key_set_counter.most_common(5):
    print(f"  {count:,} records with keys: {sorted(keys)}")


# This code will print 5 most common schemas.
# eg:
# 95,000 records → schema A
# 4,000 records  → schema B
# 1,000 records  → schema C

Most common key-sets (top 5):
  4,880,181 records with keys: ['asin', 'helpful_vote', 'images', 'parent_asin', 'rating', 'text', 'timestamp', 'title', 'user_id', 'verified_purchase']


In [14]:

if len(key_set_counter) > 1:
    all_keys = set().union(*key_set_counter.keys()) # This collects every field that appears anywhere.
    most_common_keys = key_set_counter.most_common(1)[0][0] # This stores the set of keys from the most common schema.

    # Which fields exist somewhere but aren't in the most common schema?
    missing_from_common = all_keys - most_common_keys

    if missing_from_common:
        print(f"\nFields that are sometimes absent: {sorted(missing_from_common)}")

## Printing Sample Records

In [15]:
sample_records = []

reader = pd.read_json(path, lines=True, chunksize=CHUNK_SIZE) # We have to read json file again because the previous reader has already been exhausted by the first loop.
# If not, next time we try to access reader it will raise "IOError: seek of closed file".

for i, chunk in enumerate(reader):
    if i >= MAX_CHUNKS_FOR_SAMPLE:
        break
    sample_records.extend(chunk.to_dict(orient="records"))

In [16]:
# Set a fixed random seed (42) so the random selection is reproducible
# across runs — the same sample will be chosen every time.
random.seed(42)

# Randomly select up to 5 unique records from `sample_records`.
# If there are fewer than 5 records, select all of them.
# The result is stored in `sample`.
sample = random.sample(sample_records, min(5, len(sample_records)))


In [17]:
for i, rec in enumerate(sample):
    print(f"--- Record {i} ---")
    print(json.dumps(rec, indent=2, default=str)[:1500])
    print()

--- Record 0 ---
{
  "rating": 4,
  "title": "Fun little game",
  "text": "Tri peaks with some bonus cards to help clear the fie!d",
  "images": [],
  "asin": "B01N7QK6YN",
  "parent_asin": "B01N7QK6YN",
  "user_id": "AHAX5OZ74S6RS52TPMKIAF7KTDFA",
  "timestamp": "2021-02-15 11:31:14.318000",
  "helpful_vote": 14,
  "verified_purchase": true
}

--- Record 1 ---
{
  "rating": 1,
  "title": "Won't Import from Fuji Digital Camera",
  "text": "I didn't realize Fuji was such an obscure brand of digital camera, but this software does not recognize the photo files on my camera.  I got an error message that told me to go to my camera's manufacturer and look for new drivers.  Nope. I'm not going to do that. There are too many software packages out there that don't require this extra work on the part of the user.  Sorry, Corel, I won't make the same mistake again.",
  "images": [],
  "asin": "B000VJTL1Y",
  "parent_asin": "B000VJTL1Y",
  "user_id": "AEDTXOC3YW6O7P2UPM22VNNRF77A",
  "timestamp": 

## Rating Distribution

In [18]:
RATING_CANDIDATES = ["rating", "overall", "stars"]
# We are not assuming that the rating field is always named "rating". It could be "overall" or "stars" in some datasets. So we check for these common alternatives when trying to identify the rating field in the dataset.

In [19]:
rating_col = next(
    (
        c for c in all_columns_seen # all_columns_seen is a set()
        if c.lower() in RATING_CANDIDATES
    ),
    None
)

In [20]:
rating_counts = Counter()

if rating_col:  # Works only if a rating-like column was found
    reader = pd.read_json(path, lines=True, chunksize=CHUNK_SIZE)
    for chunk in reader:
        if rating_col in chunk.columns:
            rating_counts.update(   chunk[rating_col].dropna().tolist()    )

    print(f"Using rating column: '{rating_col}'")
    for value in sorted(rating_counts):
        print(f"  {value}: {rating_counts[value]:,}")
else:
    print("No rating-like column found among:", RATING_CANDIDATES)

Using rating column: 'rating'
  1: 695,854
  2: 239,253
  3: 419,356
  4: 857,082
  5: 2,668,636


## Missing / null / empty text Check

In [21]:
TEXT_CANDIDATES = ["text", "reviewtext", "description", "review_body"]
# Again, we're not assuming the exact name.

In [22]:
text_col = next(
    (
        c for c in all_columns_seen
        if c.lower() in TEXT_CANDIDATES
    ),
    None
)

In [23]:
null_text_count = 0
empty_text_count = 0

In [24]:
if text_col:
    reader = pd.read_json(path, lines=True, chunksize=CHUNK_SIZE)
    for chunk in reader:
        if text_col in chunk.columns:
            null_text_count += chunk[text_col].isna().sum()
            empty_text_count += (chunk[text_col].astype(str).str.strip() == "").sum()

    print(f"Using text column: '{text_col}'")
    print(f"Null values: {null_text_count:,} ({null_text_count/total_rows:.2%} of all rows)")
    print(f"Empty-string values: {empty_text_count:,} ({empty_text_count/total_rows:.2%} of all rows)")
else:
    print("No text-like column found among:", TEXT_CANDIDATES)

Using text column: 'text'
Null values: 0 (0.00% of all rows)
Empty-string values: 324 (0.01% of all rows)


### Findings: The text column contains no null values, but 324 records contain empty or whitespace-only text values.

## Language + Malformed Estimate 

In [25]:
if text_col and sample_records:
    from langdetect import detect, LangDetectException

    texts = [r.get(text_col) for r in sample_records if r.get(text_col)]
    texts_sample = random.sample(texts, min(1000, len(texts)))

    def is_english(t):
        try:
            return detect(str(t)) == "en"
        except LangDetectException:
            return None

    results = [is_english(t) for t in texts_sample]
    non_english = sum(1 for r in results if r is False)
    undetectable = sum(1 for r in results if r is None)

    print(f"Sample size: {len(texts_sample)}")
    print(f"Estimated non-English: {non_english} ({non_english/len(texts_sample):.1%})")
    print(f"Undetectable (too short/ambiguous): {undetectable} ({undetectable/len(texts_sample):.1%})")

    html_tag_count = sum(1 for t in texts_sample if re.search(r"<[^>]+>", str(t)))
    print(f"Records containing HTML tags: {html_tag_count} ({html_tag_count/len(texts_sample):.1%})")
else:
    print("Skipped — no text column or no sample available.")

Sample size: 1000
Estimated non-English: 106 (10.6%)
Undetectable (too short/ambiguous): 1 (0.1%)
Records containing HTML tags: 95 (9.5%)


In [26]:
# To print a few examples of records that contain HTML tags, we can filter the `texts_sample` list for those that match the regex pattern for HTML tags. Then, we can print the first 10 examples.

html_examples = [
    t for t in texts_sample
    if re.search(r"<[^>]+>", str(t))
]

for i, text in enumerate(html_examples[:10]):
    print(f"\n--- Example {i+1} ---")
    print(text)


--- Example 1 ---
Almost annoyingly addictive! A game that needs the in app purchase, but reasonable cost if you pay attention,; with winnable levels without if you're patient...Pretty graphics, easy game<br /><br />when you're sleepy...I'm cheerfully hooked. Sigh.

--- Example 2 ---
Kindle Fire HDX, 7&#34; screen.<br /><br />I &#34;played&#34; the free version, and I noticed that the author had released a &#34;pro&#34; version. At the time I purchased this &#34;pro&#34; version, there was not much difference, it was just lacking advertisements. But I like supporting independent developers, so even with no indication that the author would improve this, I was happy knowing the small amount of change I spent went directly to the developer himself. Even if it's just to show gratitude for the &#34;free&#34; version. See, I'm a dinosaur who remembers the era of &#34;shareware&#34; games, and this particular app strikes a chord that hearkened back to those days. I suspect the developer can 

# Summary

In [27]:
print("=" * 50)
print("EXPLORATION SUMMARY")
print("=" * 50)

print(f"File: {RAW_PATH}")
print(f"Total rows: {total_rows:,}  (read in {chunk_count} chunks of {CHUNK_SIZE:,})")
print(f"Distinct key-sets: {len(key_set_counter)} (1 = fully consistent schema)")
print(f"Rating column: {rating_col or 'NOT FOUND'}")
print(f"Text column: {text_col or 'NOT FOUND'}")

if text_col:
    print(f"Null/empty text: {null_text_count + empty_text_count:,} rows ({(null_text_count+empty_text_count)/total_rows:.3%})")
if not review_hits or not text_col or not rating_col:
    print()
    print(">>> This file does not look like review data. Confirm the correct file before proceeding. <<<")

EXPLORATION SUMMARY
File: ../../data/raw/Software.jsonl
Total rows: 4,880,181  (read in 977 chunks of 5,000)
Distinct key-sets: 1 (1 = fully consistent schema)
Rating column: rating
Text column: text
Null/empty text: 324 rows (0.007%)
